In [1]:
import networkx as nx
import numpy as np
import pandas as pd

# Define the number of bus stops and junctions
num_stops = 10
num_junctions = 5  # Assume 5 junctions for now

# Generate random passenger counts for bus stops
np.random.seed(42)  # Ensuring reproducibility
passenger_counts = np.random.randint(5, 50, size=num_stops)

# Create a graph for the bus network
bus_graph = nx.Graph()

# Add bus stops as nodes with passenger counts
for i in range(num_stops):
    bus_graph.add_node(f"Stop_{i+1}", passengers=passenger_counts[i])

# Add junctions as nodes with zero passengers
for j in range(num_junctions):
    bus_graph.add_node(f"Junction_{j+1}", passengers=0)

# Define road connections (edges) between stops and junctions
edges = [
    ("Stop_1", "Junction_1"), ("Junction_1", "Stop_2"),
    ("Stop_2", "Junction_2"), ("Junction_2", "Stop_3"),
    ("Stop_3", "Junction_3"), ("Junction_3", "Stop_4"),
    ("Stop_4", "Junction_4"), ("Junction_4", "Stop_5"),
    ("Stop_5", "Junction_5"), ("Junction_5", "Stop_6"),
    ("Stop_6", "Stop_7"), ("Stop_7", "Stop_8"),
    ("Stop_8", "Stop_9"), ("Stop_9", "Stop_10")
]

# Add edges to the graph
bus_graph.add_edges_from(edges)

# Convert the graph to a DataFrame for easy visualization
bus_data = pd.DataFrame(
    [(node, data['passengers']) for node, data in bus_graph.nodes(data=True)],
    columns=["Node", "Passengers"]
)

print(bus_data)

          Node  Passengers
0       Stop_1          43
1       Stop_2          33
2       Stop_3          19
3       Stop_4          47
4       Stop_5          12
5       Stop_6          25
6       Stop_7          43
7       Stop_8          23
8       Stop_9          27
9      Stop_10          15
10  Junction_1           0
11  Junction_2           0
12  Junction_3           0
13  Junction_4           0
14  Junction_5           0


In [ ]:
import time
import random

def update_passenger_counts(graph):
    """Simulates real-time changes in passenger counts at bus stops."""
    while True:
        for node in graph.nodes:
            if "Stop" in node:
                # Randomly increase or decrease passenger count
                change = random.randint(-3, 5)  # Some people leave, others arrive
                graph.nodes[node]['passengers'] = max(0, graph.nodes[node]['passengers'] + change)

        # Print current passenger counts (can be logged into a database)
        print({node: graph.nodes[node]['passengers'] for node in graph.nodes if "Stop" in node})

        time.sleep(5)  # Simulate updates every 5 seconds

# Run the simulation (can be adapted for real-time integration)
update_passenger_counts(bus_graph)


{'Stop_1': np.int64(48), 'Stop_2': np.int64(30), 'Stop_3': np.int64(16), 'Stop_4': np.int64(48), 'Stop_5': np.int64(17), 'Stop_6': np.int64(22), 'Stop_7': np.int64(46), 'Stop_8': np.int64(22), 'Stop_9': np.int64(29), 'Stop_10': np.int64(17)}
{'Stop_1': np.int64(53), 'Stop_2': np.int64(28), 'Stop_3': np.int64(13), 'Stop_4': np.int64(49), 'Stop_5': np.int64(15), 'Stop_6': np.int64(22), 'Stop_7': np.int64(48), 'Stop_8': np.int64(24), 'Stop_9': np.int64(33), 'Stop_10': np.int64(16)}
{'Stop_1': np.int64(55), 'Stop_2': np.int64(33), 'Stop_3': np.int64(12), 'Stop_4': np.int64(48), 'Stop_5': np.int64(12), 'Stop_6': np.int64(22), 'Stop_7': np.int64(52), 'Stop_8': np.int64(22), 'Stop_9': np.int64(34), 'Stop_10': np.int64(17)}
{'Stop_1': np.int64(56), 'Stop_2': np.int64(35), 'Stop_3': np.int64(11), 'Stop_4': np.int64(47), 'Stop_5': np.int64(9), 'Stop_6': np.int64(25), 'Stop_7': np.int64(55), 'Stop_8': np.int64(27), 'Stop_9': np.int64(34), 'Stop_10': np.int64(14)}
{'Stop_1': np.int64(59), 'Stop_2'

In [ ]:
!pip install pymongo dnspython


In [ ]:
import numpy as np
import pandas as pd
import random
from datetime import datetime, timedelta

# Parameters
days = 5 * 365  # Generate data for 5 years
buses_per_day = 100  # Number of bus trips per day

# Define base demand pattern (higher in peak hours, lower at night)
def passenger_demand(hour, is_weekend, season_factor):
    base_demand = {
        0: 10, 1: 5, 2: 3, 3: 2, 4: 5, 5: 15, 6: 30, 7: 50, 8: 80, 9: 60, 10: 50,
        11: 40, 12: 50, 13: 55, 14: 60, 15: 70, 16: 90, 17: 120, 18: 100, 19: 80, 20: 60, 21: 40, 22: 20, 23: 15
    }
    demand = base_demand[hour] * season_factor
    if is_weekend:
        demand *= 0.7  # Lower demand on weekends
    return max(2, int(demand + random.gauss(0, 10)))

# Generate data
data = []
start_date = datetime(2020, 1, 1)
for day in range(days):
    date = start_date + timedelta(days=day)
    is_weekend = date.weekday() >= 5  # Saturday & Sunday
    season_factor = 1.2 if date.month in [6, 7, 8] else (0.8 if date.month in [12, 1, 2] else 1.0)

    for _ in range(buses_per_day):
        hour = random.randint(0, 23)
        passengers = passenger_demand(hour, is_weekend, season_factor)
        data.append([date.strftime('%Y-%m-%d'), hour, passengers])

# Convert to DataFrame
df = pd.DataFrame(data, columns=['Date', 'Hour', 'Passengers'])

# Save to CSV
df.to_csv("realistic_passenger_data.csv", index=False)
print("Generated dataset with", len(df), "entries.")


In [ ]:
!pip install pandas numpy matplotlib scikit-learn tensorflow

In [ ]:
import pandas as pd

# Load the dataset
file_path = "/content/realistic_passenger_data.csv"  # Update this path if needed
df = pd.read_csv(file_path)

# Display first few rows
print(df.head())

# Check dataset information
print(df.info())
print(df.columns)


In [ ]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# Step 1: Normalize the passenger count data
scaler = MinMaxScaler(feature_range=(0, 1))
df["Passengers"] = scaler.fit_transform(df["Passengers"].values.reshape(-1, 1))

# Step 2: Create sequences for LSTM training
def create_sequences(data, time_steps=10):
    X, y = [], []
    for i in range(len(data) - time_steps):
        X.append(data[i : i + time_steps])
        y.append(data[i + time_steps])
    return np.array(X), np.array(y)

time_steps = 10  # Define number of past steps to consider
X, y = create_sequences(df["Passengers"].values, time_steps)

# Step 3: Split into training and testing sets
train_size = int(len(X) * 0.8)  # 80% training data
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# Reshape for LSTM (samples, time_steps, features)
X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

print("Data Preprocessing Complete! Shapes:")
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

# Define the LSTM model
model = Sequential([
    LSTM(50, return_sequences=True, input_shape=(X_train.shape[1], 1)),  # First LSTM layer
    LSTM(50, return_sequences=False),  # Second LSTM layer
    Dense(25, activation="relu"),  # Dense layer with ReLU activation
    Dense(1)  # Output layer
])

# Compile the model
model.compile(optimizer="adam", loss="mse")

# Display model summary
model.summary()


In [ ]:
from sklearn.model_selection import train_test_split

# Split into training (80%) and validation (20%) sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, shuffle=False)

print("Training data shape:", X_train.shape, y_train.shape)
print("Validation data shape:", X_val.shape, y_val.shape)

# Train the LSTM model
history = model.fit(X_train, y_train,
                    batch_size=32,
                    epochs=50,  # You can increase epochs for better performance
                    validation_data=(X_val, y_val),
                    verbose=1)

# Plot training loss
import matplotlib.pyplot as plt

plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.title('Training vs Validation Loss')
plt.show()

In [ ]:

loss, accuracy = model.evaluate(X_val, y_val, verbose=0)
print(f"Validation Accuracy: {accuracy:.4f}")

In [ ]:
model.save('my_lstm_model.h5')  # Save as HDF5 file